Exploring an example of an image from the ISIC dataset

Imports

In [36]:
from PIL import Image
import os
import pandas as pd
from pathlib import Path


Switching the working directory so that I can reference from the project root

In [5]:
os.chdir("..")   # moves up one level, to the project root
print(os.getcwd())

/Users/evangoldstein/python-projects/Melanoma-Classifier/melanoma-classifier


Extracting the first image from the ISIC dataset that is not downsampled

In [19]:
ex1 = Image.open("data/raw/isic/ISIC_2019_Training_Input/ISIC_2019_Training_Input/ISIC_0000000.jpg")

In [20]:
print(ex1.format,ex1.size,ex1.mode)

JPEG (1022, 767) RGB


Extracting the first downsampled image from the ISC dataset and comparing its format, size, and mode witi the 1 non-downsampled image

In [21]:
ex2 = Image.open("data/raw/isic/ISIC_2019_Training_Input/ISIC_2019_Training_Input/ISIC_0000017_downsampled.jpg")
print(f'Is the non-downsmapled image format the same as the downsampled: {ex1.format==ex2.format}')
print(f'Is the non-downsmapled image size the same as the downsampled: {ex1.size==ex2.size}, the sizes are: non-downsampled {ex1.size} and downsampled {ex2.size}')
print(f'Is the non-downsmapled image mode the same as the downsampled: {ex1.mode==ex2.mode}')

Is the non-downsmapled image format the same as the downsampled: True
Is the non-downsmapled image size the same as the downsampled: False, the sizes are: non-downsampled (1022, 767) and downsampled (1024, 768)
Is the non-downsmapled image mode the same as the downsampled: True


Testing on another non-downsampled image

In [22]:
ex3 = Image.open("data/raw/isic/ISIC_2019_Training_Input/ISIC_2019_Training_Input/ISIC_0000001.jpg")
print(f'Is the non-downsmapled image format the same as the downsampled: {ex1.format==ex3.format}')
print(f'Is the non-downsmapled image size the same as the downsampled: {ex1.size==ex3.size}, the sizes are: non-downsampled {ex1.size} and downsampled {ex3.size}')
print(f'Is the non-downsmapled image mode the same as the downsampled: {ex1.mode==ex3.mode}')

Is the non-downsmapled image format the same as the downsampled: True
Is the non-downsmapled image size the same as the downsampled: True, the sizes are: non-downsampled (1022, 767) and downsampled (1022, 767)
Is the non-downsmapled image mode the same as the downsampled: True


It is evident that atleast the first two non-downsampled images are the same in terms of format, size, and mode. Now I am going to check if the other downsampled images are the same in terms of these qualities as well.

In [23]:
ex4 = Image.open("data/raw/isic/ISIC_2019_Training_Input/ISIC_2019_Training_Input/ISIC_0000019_downsampled.jpg")
print(f'Is the non-downsmapled image format the same as the downsampled: {ex4.format==ex2.format}')
print(f'Is the non-downsmapled image size the same as the downsampled: {ex4.size==ex2.size}, the sizes are: non-downsampled {ex4.size} and downsampled {ex2.size}')
print(f'Is the non-downsmapled image mode the same as the downsampled: {ex4.mode==ex2.mode}')

Is the non-downsmapled image format the same as the downsampled: True
Is the non-downsmapled image size the same as the downsampled: True, the sizes are: non-downsampled (1024, 768) and downsampled (1024, 768)
Is the non-downsmapled image mode the same as the downsampled: True


It appears at the surface that all the downsampled images are the same in terms of format, size, and mode while the non-downsampled are also the same in these quantities but differ from their downsampled counterparts. I am going to attempt to get the format, size, and mode for each image in the dataset to test this theory

Reading in the meta for the image data to obtain the file names an get calculated columns from the image

Writing a get header method to obtain the format, size, and mode of the image and avoid calling Image.open() multiple times

In [ ]:
root = Path("data/raw/isic/ISIC_2019_Training_Input/ISIC_2019_Training_Input")

EMPTY_img_detail_header = {"image_format": None, "image_size": None, "image_mode": None, "error": None}
def get_header(img) -> dict:
    """
    This method gets an image name and opens it by appending on it root folder and file type and returns an exploratory dictionary
    to allow one to see if a file is None and why in addition to being able to analyze the the data relating to the image

    Args:
        img str: a string representing the image name stored in the directory
    
    Returns:
        dict: a dictionary with the image format, image size, image mode if they exist, dictionary is preloaded with None values in
        addition to the type of error if one exist

    Raises:
        FileNotFoundError: if the file does not exist in the directory
        OSError, SyntaxError: if the file is unreadable
    """
    img_path = root / f"{img}.jpg"
    try:
        with Image.open(img_path) as im:
            return {**EMPTY_img_detail_header,"image_format": im.format, "image_size": im.size,
                    "image_mode": im.mode, "status": "ok"}
    except FileNotFoundError:
        return {**EMPTY_img_detail_header, "status": "missing"}
    except (OSError, SyntaxError) as e:
        return {**EMPTY_img_detail_header, "status": "unreadable", "error": str(e)}


Testing the header method

In [61]:
print(get_header('ISIC_0000000'))

{'image_format': 'JPEG', 'image_size': (1022, 767), 'image_mode': 'RGB', 'error': None, 'status': 'ok'}


Reading in the metadata file

In [ ]:
meta_data = pd.read_csv("data/raw/isic/ISIC_2019_Training_Metadata.csv")

Getting the headers which include the image format, size, mode, and whether errors have occured.

In [63]:
img_detail_header = pd.DataFrame([get_header(img) for img in list(meta_data["image"])], columns=EMPTY_img_detail_header.keys(),index=meta_data.index)

Looking at if errors are present and below they appear not to be as the number error count of images is equal to the number of images

In [66]:
img_detail_header.isna().value_counts()

image_format  image_size  image_mode  error
False         False       False       True     25331
Name: count, dtype: int64

Now I am going to explore the downsampled data and see if the number of downsampled matches the 2074 of which I found earlier

In [73]:
print(meta_data["image"].str.contains("downsampled").sum())

2074


Now I am going to explore class imbalances as this will help in splitting the data later for training

In [ ]:
print(f'The number of na in the lesion_id are: {meta_data["lesion_id"].isna().sum()}')
print(f'The number of not na in the lesion_id are: {meta_data["lesion_id"].notna().sum()}')
print(f'The number of unique {meta_data["lesion_id"].nunique()}')

2084
23247
11847


On avearge here one can see that each lesion has about 2 images which is something to keep in mind as this could be a potential data leakage if one these images end up in the training in the other in the testing set. I am going to need to create a data structure to store the images that belong with each lesion_id and only sample from the lesion_id to prevent this leakage problem

In [75]:
print(f'I want to determine the number of images assoicated with each lesion_id and how many of these unique counts there are. The counts are as following')
print(meta_data["lesion_id"].value_counts().value_counts().sort_index())

I want to determine the number of images assoicated with each lesion_id and how many of these unique counts there are. The counts are as following
count
1     6788
2     2505
3     1306
4      468
5      248
6      198
7      112
8       59
9       54
10      24
11      18
12      19
13      10
14       7
15       3
16       3
17       5
18       3
19       6
20       6
21       1
24       1
26       1
27       1
31       1
Name: count, dtype: int64


As shown above some of the lesion's have many images assoicated with them and therefore I need to be careful when sampling for training and testing. The multiple images could be a historical progression of the lesion in question or from different angles.

In [76]:
lesion_counts = meta_data['lesion_id'].value_counts()
print(lesion_counts.is_monotonic_decreasing)
print(lesion_counts.max())
print(lesion_counts.idxmax())

True
31
BCN_0001728


Looking into the lesion with 31 lesions

In [77]:
lesion_count_31 = meta_data[meta_data["lesion_id"] == lesion_counts.idxmax()]
print(lesion_count_31)

              image  age_approx anatom_site_general    lesion_id   sex
13152  ISIC_0053824        50.0      anterior torso  BCN_0001728  male
14815  ISIC_0056537        50.0      anterior torso  BCN_0001728  male
15054  ISIC_0056915        50.0      anterior torso  BCN_0001728  male
15557  ISIC_0057744        50.0      anterior torso  BCN_0001728  male
15609  ISIC_0057828        50.0      anterior torso  BCN_0001728  male
15666  ISIC_0057918        50.0      anterior torso  BCN_0001728  male
16280  ISIC_0058876        50.0      anterior torso  BCN_0001728  male
16738  ISIC_0059595        50.0      anterior torso  BCN_0001728  male
16790  ISIC_0059684        50.0      anterior torso  BCN_0001728  male
17951  ISIC_0061544        50.0      anterior torso  BCN_0001728  male
18029  ISIC_0061660        50.0      anterior torso  BCN_0001728  male
18031  ISIC_0061664        50.0      anterior torso  BCN_0001728  male
18119  ISIC_0061806        50.0      anterior torso  BCN_0001728  male
18553 

Before I go any further I need to look at the ground_truth data so that I can expore this alongside the metadata to answer more questions about the meta data

In [78]:
classes = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC", "UNK"]
labels = pd.read_csv("data/raw/isic/ISIC_2019_Training_GroundTruth.csv")
print(labels[classes].sum(axis=1).value_counts())

1.0    25331
Name: count, dtype: int64


It appear this is not a multi-label classification problem but rather a multiclass classification problem. Now I know each image only has one label associated with it

I am not going to look at each column more thoroughly. First I am going to examen the age_approx. I am going to guess there will be a few images without approximate ages

In [ ]:
age_approx = meta_data['age_approx']
print(age_approx.describe)
print(f'There exist {age_approx.isna().sum()} images without ages, or rougly 1/60th of the data')
print(f'There exist {age_approx.notna().sum()} image with ages, or roughly 59/60th of the data so most of the data')

<bound method Series.describe of 0        55.0
1        30.0
2        60.0
3        30.0
4        80.0
         ... 
25326    85.0
25327    65.0
25328    70.0
25329    55.0
25330    50.0
Name: age_approx, Length: 25331, dtype: float64>
There exist 437 images without ages, or rougly 1/60th of the data
There exist 24894 image with ages, or roughly 59/60 % of the data so most of the data


Next i am going to look at the antom_site_general

In [ ]:
anatom_site_general = meta_data["anatom_site_general"]
print("Value counts")
print(anatom_site_general.value_counts())
print("Analyzing if any data is na or not")
print(f'There exist {anatom_site_general.isna().sum()} images without anatom sites, or rougly 1/10th of the data')
print(f'There exist {anatom_site_general.notna().sum()} image with anatom sites, or roughly 9/10th of the data so most of the data but still a troubling amount to look into')

Value counts
anatom_site_general
anterior torso     6915
lower extremity    4990
head/neck          4587
upper extremity    2910
posterior torso    2787
palms/soles         398
oral/genital         59
lateral torso        54
Name: count, dtype: int64
Analyzing if any data is na or not
There exist 2631 images without anatom sites, or rougly 1/10th of the data
There exist 22700 image with anatom sites, or roughly 9/10th of the data so most of the data


Finally moving onto the sex column I am going to expect to see most of the images have a sex listed.

In [87]:
sex = meta_data["sex"]
print(sex.value_counts(dropna=False))
print("roughly 1/60th of the images don't have a sex attached to them and 59/60 do. In this data it only list 2 sexes as well")

sex
male      13286
female    11661
NaN         384
Name: count, dtype: int64
roughly 1/60th of the images don't have a sex attached to them and 59/60 do. In this data it only list 2 sexes as well


Gathering what I have found exploring each of these columns. It is clear that the anatom_site_gernal column will be the biggest issue as around 10% of the data is missing this data. There may exist some method to interpolate this data otherwise if this data is uniform across the classes I can maybe considering droping the ones I can't interpolate. Right now we are at max looking at most 2631 images to be NA but i suspect that many of the images that have one NA have many NA's and now I am going to check the distribution of the missing data.

In [91]:
print(meta_data.isna().sum(axis=1).value_counts())

0    21311
1     2983
2      709
3      177
4      151
Name: count, dtype: int64


From here I see that there is around 4000 images that have at least one missing value which is around 1/6 of the data. That is quite a bit of data and needs some more looking into. For example some of the images have 3 to 4 pieces of missing data and this makes up about 1/60th of the entire data set so it could be feasible to just drop this data altother or more so ensure that when sampling for the test and training sets there is a balance of where these go so the model can learn.